# R4 · Smoke test Qwen3.5-2B (Tuần 1, ngày 1–2)

Mục tiêu (Plan4project A4, C4, C7):
1. Load được `Qwen/Qwen3.5-2B` trên GPU T4 (Colab/Kaggle), **chỉ dùng văn bản**.
2. Xác nhận **thinking đang TẮT** (không sinh `<think>…</think>` hoặc khối suy luận).
3. Chạy thử 4 prompt nháp `basic` / `strict` (VI & EN) với greedy, `max_new_tokens=256`, đo tốc độ.
4. Thử hàm hậu xử lý output (C7).

**Từ model card (đã đọc 11/2026):**
- Repo: `Qwen/Qwen3.5-2B` (bản post-trained; base là `Qwen/Qwen3.5-2B-Base`, không dùng). License Apache-2.0.
- Class: `AutoModelForMultimodalLM` + `AutoProcessor`. Cần `transformers` bản mới nhất (model card gợi ý cài từ nhánh `main`).
- *"Qwen3.5-2B operates in non-thinking mode by default"*. Soft switch `/think`, `/nothink` của Qwen3 **không** được hỗ trợ.
- Kiến trúc lai Gated DeltaNet + Gated Attention, 24 layer, context 262K.

**Cách dùng:** Runtime → GPU T4. Chạy lần lượt từ trên xuống. Ghi kết quả ô cuối vào `docs/r4_qwen_notes.md`.

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


## 1. Cài thư viện
Model card yêu cầu `transformers` mới nhất. Thử bản PyPI trước; nếu ô load ở bước 2 báo không nhận kiến trúc `qwen3_5`, bỏ comment dòng cài từ `main` rồi **Restart runtime**.

In [2]:
!pip install -q -U transformers accelerate
# !pip install -q -U "git+https://github.com/huggingface/transformers.git@main" accelerate
import torch, transformers, sys
print("python", sys.version.split()[0], "| torch", torch.__version__, "| transformers", transformers.__version__)
print("cuda:", torch.cuda.is_available(), "| bf16 supported:", torch.cuda.is_available() and torch.cuda.is_bf16_supported())

python 3.13.16 | torch 2.14.1+cu130 | transformers 5.24.1
cuda: True | bf16 supported: True


## 2. Load model
- T4 không hỗ trợ bf16 thật sự → dùng `float16`. Nếu output ra rỗng / toàn ký tự lạ / NaN (tràn số fp16), đổi `DTYPE = torch.float32` (2B × 4 byte ≈ 8–9 GB, vẫn vừa T4 16 GB).
- Thử lần lượt các class: `AutoModelForMultimodalLM` (theo model card) → `AutoModelForImageTextToText` → `AutoModelForCausalLM`. Ghi lại class nào chạy được.

In [3]:
import time, torch, transformers
from transformers import AutoProcessor, AutoTokenizer

MODEL_ID = "Qwen/Qwen3.5-2B"   # dự phòng (C4): "Qwen/Qwen2.5-1.5B-Instruct"
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() and torch.cuda.get_device_capability()[0] >= 8 else torch.float16
# DTYPE = torch.float32   # bật nếu fp16 cho output hỏng

model, LOADED_CLASS, errors = None, None, {}
for cls_name in ["AutoModelForMultimodalLM", "AutoModelForImageTextToText", "AutoModelForCausalLM"]:
    cls = getattr(transformers, cls_name, None)
    if cls is None:
        errors[cls_name] = "không có trong phiên bản transformers này"
        continue
    try:
        t0 = time.time()
        model = cls.from_pretrained(MODEL_ID, dtype=DTYPE, device_map="auto")
        LOADED_CLASS = cls_name
        print(f"OK: {cls_name} ({time.time()-t0:.0f}s), dtype={DTYPE}")
        break
    except Exception as e:
        errors[cls_name] = f"{type(e).__name__}: {e}"
for k, v in errors.items():
    print(f"[lỗi] {k}: {v[:300]}")
assert model is not None, "Không load được model -> thử cài transformers@main (bước 1) hoặc phương án dự phòng C4"
model.eval()

try:
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    tokenizer = getattr(processor, "tokenizer", processor)
except Exception as e:
    print("AutoProcessor lỗi, dùng AutoTokenizer:", e)
    processor = tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("model type:", model.config.model_type, "| class thực tế:", type(model).__name__)
print("GPU mem đã dùng: %.2f GB" % (torch.cuda.memory_allocated() / 1e9))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/2.91k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/64.5k [00:00<?, ?B/s]

model.safetensors-00001-of-00001.safeten(…): reconstructing file:   0%|          |  0.00B / 4.55GB            

model.safetensors-00001-of-00001.safeten(…): downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

... (4 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

model type: qwen3_5 | class thực tế: Qwen3_5ForConditionalGeneration
GPU mem đã dùng: 4.43 GB


## 3. Kiểm tra chat template & chế độ thinking
So sánh prompt sau khi áp template ở 3 trường hợp: mặc định, `enable_thinking=False`, `enable_thinking=True`.
Kỳ vọng: **mặc định == False** (model card nói 2B mặc định non-thinking). Trong `run_llm.py` vẫn truyền `enable_thinking=False` tường minh cho chắc.

In [4]:
def build_messages(user_text):
    # Chỉ văn bản, không ảnh (C4). Dạng list-of-parts để processor đa phương thức nhận đúng.
    return [{"role": "user", "content": [{"type": "text", "text": user_text}]}]

def render(user_text, **kw):
    return tokenizer.apply_chat_template(build_messages(user_text), tokenize=False,
                                         add_generation_prompt=True, **kw)

probe = "Xin chào"
variants = {}
for name, kw in [("default", {}), ("think=False", {"enable_thinking": False}), ("think=True", {"enable_thinking": True})]:
    try:
        variants[name] = render(probe, **kw)
    except Exception as e:
        variants[name] = f"<LỖI {type(e).__name__}: {e}>"
for name, s in variants.items():
    print(f"===== {name} =====\n{s!r}\n")
print("default == think=False :", variants["default"] == variants["think=False"])
print("default == think=True  :", variants["default"] == variants["think=True"])

===== default =====
'<|im_start|>user\nXin chào<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'

===== think=False =====
'<|im_start|>user\nXin chào<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n'

===== think=True =====
'<|im_start|>user\nXin chào<|im_end|>\n<|im_start|>assistant\n<think>\n'

default == think=False : True
default == think=True  : False


## 4. Sinh thử & phát hiện thinking trong output
Greedy, `max_new_tokens=256` theo C1. (Model card khuyên sampling `temperature=1.0, top_k=20, presence_penalty=2.0` cho non-thinking, nhưng dự án cố định greedy để tái lập — ghi rõ lựa chọn này trong báo cáo.)

In [5]:
THINK_MARKERS = ["<think>", "</think>", "Thinking Process", "<|im_start|>think"]

@torch.inference_mode()
def generate(user_text, enable_thinking=False, max_new_tokens=256):
    text = render(user_text, enable_thinking=enable_thinking)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    t0 = time.time()
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                         temperature=None, top_p=None, top_k=None,
                         pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
    dt = time.time() - t0
    new = out[0, inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(new, skip_special_tokens=False)
    clean = tokenizer.decode(new, skip_special_tokens=True)
    return {"raw": raw, "clean": clean, "n_tokens": len(new), "sec": dt,
            "has_think": any(m in raw for m in THINK_MARKERS)}

for q in ["Thủ đô của Việt Nam là gì? Trả lời một câu.", "What is 17 * 23? Answer with the number only."]:
    r = generate(q, enable_thinking=False)
    print(f"Q: {q}\nraw: {r['raw']!r}\nhas_think={r['has_think']} | {r['n_tokens']} tok / {r['sec']:.1f}s\n")

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


Q: Thủ đô của Việt Nam là gì? Trả lời một câu.
raw: 'Hà Nội<|im_end|>\n<|endoftext|>'
has_think=False | 5 tok / 2.1s

Q: What is 17 * 23? Answer with the number only.
raw: '391<|im_end|>\n<|endoftext|>'
has_think=False | 6 tok / 0.3s



In [6]:
# Đối chứng: bật thinking để biết thinking "trông như thế nào" -> bộ phát hiện ở trên có bắt được không.
r = generate("What is 17 * 23?", enable_thinking=True, max_new_tokens=200)
print("raw (think=True):", repr(r["raw"][:600]))
print("has_think:", r["has_think"])

raw (think=True): 'Thinking Process:\n\n1.  **Analyze the Request:** The user is asking for the product of two numbers: 17 and 23.\n\n2.  **Perform the Calculation:**\n    *   Method 1: Standard multiplication.\n        *   $17 \\times 23$\n        *   $17 \\times 3 = 51$\n        *   $17 \\times 20 = 340$\n        *   $340 + 51 = 391$\n    *   Method 2: Breakdown.\n        *   $(20 - 3) \\times 17$\n        *   $20 \\times 17 - 3 \\times 17$\n        *   $340 - 51 = 289$\n        *   Wait, $23 = 20 + 3'
has_think: True


## 5. Prompt nháp basic / strict + hậu xử lý (C7)
Prompt đọc từ `prompts/*.txt` trong repo. Thay `{source}` bằng `str.replace` (không dùng `.format` vì dòng OCR có thể chứa `{}`).
Nếu chưa clone repo, ô dưới sẽ clone; hoặc upload thư mục `prompts/` lên.

In [9]:
import os, re, pathlib
REPO_URL = "https://github.com/duc130406cp/fin-ocrfix.git"
if not pathlib.Path("prompts").exists() and not pathlib.Path("fin-ocrfix/prompts").exists():
    os.system(f"git clone -q {REPO_URL}")
PROMPT_DIR = pathlib.Path("prompts") if pathlib.Path("prompts").exists() else pathlib.Path("fin-ocrfix/prompts")
PROMPTS = {p.stem: p.read_text(encoding="utf-8") for p in sorted(PROMPT_DIR.glob("*.txt"))}
assert PROMPTS, "Không tìm thấy thư mục prompts/ (nếu repo đang private, hãy upload thư mục prompts/ lên Colab hoặc dùng token để clone)."
print(list(PROMPTS))

PREFIX_RE = re.compile(r"^\s*(Dòng đã sửa|Dong da sua|Corrected line|Corrected|Output)\s*:\s*", re.I)
QUOTES = [('"', '"'), ("“", "”"), ("'", "'"), ("«", "»"), ("`", "`")]

def postprocess(output: str):
    """C7: lấy dòng đầu tiên không rỗng, bỏ tiền tố kiểu 'Dòng đã sửa:', bỏ ngoặc kép bao ngoài.
    Trả về (prediction, format_error). format_error=True nếu phải can thiệp / output bất thường."""
    import unicodedata
    text = re.sub(r"<think>.*?</think>", "", output, flags=re.S)  # phòng hờ
    lines = [l for l in text.splitlines() if l.strip()]
    if not lines:
        return "", True
    line, fmt_err = lines[0].strip(), len(lines) > 1
    m = PREFIX_RE.match(line)
    if m:
        line, fmt_err = line[m.end():].strip(), True
    for a, b in QUOTES:
        if len(line) >= 2 and line.startswith(a) and line.endswith(b):
            line, fmt_err = line[1:-1].strip(), True
            break
    return unicodedata.normalize("NFC", line), fmt_err

# unit test nhỏ cho hậu xử lý
assert postprocess("Doanh thu thuần") == ("Doanh thu thuần", False)
assert postprocess("\n\nDòng đã sửa: \"Doanh thu thuần\"\nGiải thích...") == ("Doanh thu thuần", True)
assert postprocess("Corrected line: Net revenue") == ("Net revenue", True)
assert postprocess("   ") == ("", True)
print("postprocess OK")

def fill(prompt_name, source):
    return PROMPTS[prompt_name].replace("{source}", source)

['en_basic', 'en_strict', 'vi_basic', 'vi_strict']
postprocess OK


In [10]:
# Dòng thử (tự soạn, KHÔNG phải test set). Mỗi ngôn ngữ: lỗi dấu/ký tự, lỗi số, dòng đã đúng.
SAMPLES = {
    "vi": [
        ("Gia von hang ban giam do chi phi nguyen vat lieu thap hon", "Giá vốn hàng bán giảm do chi phí nguyên vật liệu thấp hơn"),
        ("Vốn chủ sở hữu tại ngày 31/12/2025 là 4.5O8.120 triệu đồng", "Vốn chủ sở hữu tại ngày 31/12/2025 là 4.508.120 triệu đồng"),
        ("Khấu hao tài sản cố định được tính theo phương pháp đường thẳng", "Khấu hao tài sản cố định được tính theo phương pháp đường thẳng"),
    ],
    "en": [
        ("Cash and cash equivaIents at the end of the year", "Cash and cash equivalents at the end of the year"),
        ("Retained earnings increased to VND 2,3l4 billion in 2025", "Retained earnings increased to VND 2,314 billion in 2025"),
        ("Cost of goods sold decreased by 8% compared to 2024", "Cost of goods sold decreased by 8% compared to 2024"),
    ],
}
rows = []
for lang, pairs in SAMPLES.items():
    for variant in ["basic", "strict"]:
        for src, tgt in pairs:
            r = generate(fill(f"{lang}_{variant}", src), enable_thinking=False)
            pred, fmt_err = postprocess(r["clean"])
            rows.append(dict(lang=lang, prompt=variant, source=src, pred=pred, exact=(pred == tgt),
                             fmt_err=fmt_err, think=r["has_think"], tok=r["n_tokens"], sec=round(r["sec"], 2)))
import pandas as pd
pd.set_option("display.max_colwidth", 80)
df = pd.DataFrame(rows); df

,lang,prompt,source,pred,exact,fmt_err,think,tok,sec
0,vi,basic,Gia von hang ban giam do chi phi nguyen vat lieu thap hon,Gia vốn hàng bán giảm do chi phí nguyên vật liệu thấp hơn,False,False,False,16,1.38
1,vi,basic,Vốn chủ sở hữu tại ngày 31/12/2025 là 4.5O8.120 triệu đồng,Vốn chủ sở hữu tại ngày 31/12/2025 là 4.508.120 triệu đồng,True,False,False,34,1.94
2,vi,basic,Khấu hao tài sản cố định được tính theo phương pháp đường thẳng,Khấu hao tài sản cố định được tính theo phương pháp đường thẳng,True,False,False,17,1.27
3,vi,strict,Gia von hang ban giam do chi phi nguyen vat lieu thap hon,Gia vốn hàng bán giảm do chi phí nguyên vật liệu thấp hơn,False,False,False,16,1.14
4,vi,strict,Vốn chủ sở hữu tại ngày 31/12/2025 là 4.5O8.120 triệu đồng,Vốn chủ sở hữu tại ngày 31/12/2025 là 4.508.120 triệu đồng,True,False,False,34,1.82
5,vi,strict,Khấu hao tài sản cố định được tính theo phương pháp đường thẳng,Khấu hao tài sản cố định được tính theo phương pháp đường thẳng,True,False,False,17,0.94
6,en,basic,Cash and cash equivaIents at the end of the year,Cash and cash equivalents at the end of the year,True,False,False,13,0.73
7,en,basic,"Retained earnings increased to VND 2,3l4 billion in 2025","Retained earnings increased to VND 2,314 billion in 2025",True,False,False,22,1.21
8,en,basic,Cost of goods sold decreased by 8% compared to 2024,Cost of goods sold decreased by 8% compared to 2023,False,False,False,19,1.06
9,en,strict,Cash and cash equivaIents at the end of the year,Cash and cash equivalents at the end of the year,True,False,False,13,0.72


## 6. Tổng kết (copy vào `docs/r4_qwen_notes.md`)

In [11]:
print(f"""- Môi trường: {torch.cuda.get_device_name(0)}, torch {torch.__version__}, transformers {transformers.__version__}
- Model: {MODEL_ID}, class load được: {LOADED_CLASS} ({type(model).__name__}), dtype {DTYPE}
- VRAM sau khi load: {torch.cuda.memory_allocated()/1e9:.2f} GB, đỉnh: {torch.cuda.max_memory_allocated()/1e9:.2f} GB
- Template mặc định == enable_thinking=False: {variants['default'] == variants['think=False']}
- Output có dấu hiệu thinking (enable_thinking=False): {int(df.think.sum())}/{len(df)} dòng
- Lỗi định dạng output: {int(df.fmt_err.sum())}/{len(df)}
- Khớp hoàn toàn target: basic {df[df.prompt=='basic'].exact.mean():.0%}, strict {df[df.prompt=='strict'].exact.mean():.0%}
- Tốc độ trung bình: {df.sec.mean():.2f} s/dòng (batch=1) -> ước tính 1.500 dòng × 2 nhiễu × 2 prompt ≈ {df.sec.mean()*6000/3600:.1f} giờ
""")

- Môi trường: Tesla T4, torch 2.14.1+cu130, transformers 5.24.1
- Model: Qwen/Qwen3.5-2B, class load được: AutoModelForMultimodalLM (Qwen3_5ForConditionalGeneration), dtype torch.float16
- VRAM sau khi load: 4.44 GB, đỉnh: 4.51 GB
- Template mặc định == enable_thinking=False: True
- Output có dấu hiệu thinking (enable_thinking=False): 0/12 dòng
- Lỗi định dạng output: 0/12
- Khớp hoàn toàn target: basic 67%, strict 83%
- Tốc độ trung bình: 1.21 s/dòng (batch=1) -> ước tính 1.500 dòng × 2 nhiễu × 2 prompt ≈ 2.0 giờ

